In [3]:
#Analysis Overview
#To address the request, the following steps were taken:
#Data Preprocessing: The dataset was loaded, and the date column was converted to a datetime object. The data was then down-sampled from daily to monthly frequency.
#For open_SAR (Open Price), the monthly mean was calculated to represent the average price level for that month.
#For volume, the monthly mean was calculated to represent the average daily trading activity for that month.
#Stationarity Check: The Augmented Dickey-Fuller (ADF) test was performed to check for stationarity. Both variables were found to be non-stationary (p-value > 0.05), indicating that differencing (analyzing the change rather than the raw value) is necessary for reliable modeling.
#Univariate Modeling (ARIMA): Two separate ARIMA models were fitted:
#Open SAR: An ARIMA(2, 1, 0) model was selected as the best fit based on AIC. This suggests the change in price depends on the changes from the previous two months.
#Volume: An ARIMA(1, 1, 0) model was selected. This suggests the change in volume depends on the change from the previous month.
#Multivariate Modeling (VAR): A Vector Autoregression (VAR) model was fitted on the differenced (stationary) data to explore the relationship between price and volume.
#The optimal lag order selected was 1.
#The results show strong auto-correlation for price (past price changes predict future price changes).
#However, no significant causal relationship was found between Volume and Price in either direction (Granger causality implied by insignificant cross-coefficients).

In [4]:
# Import necessary libraries

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.api import VAR
import warnings

In [6]:
# 1 load the data
df = pd.read_csv("dc-2.csv")
df

,date,open_SAR,open_USD,high_SAR,high_USD,low_SAR,low_USD,close_SAR,close_USD,volume
0,07-05-2018,36232.68941,9661.02,36340.13837,9689.67,34432.42240,9181.00,35122.49600,9365.00,33787
1,08-05-2018,35122.49600,9365.00,35537.66528,9475.70,33980.64922,9060.54,34457.02502,9187.56,25533
2,09-05-2018,34421.17120,9178.00,35216.25600,9390.00,33622.33600,8965.00,34916.22400,9310.00,25673
3,10-05-2018,34916.22400,9310.00,35235.45805,9395.12,33641.08800,8970.00,33761.85088,9002.20,25055
4,11-05-2018,33761.88838,9002.21,33816.60672,9016.80,31282.08640,8341.00,31503.36000,8400.00,48227
...,...,...,...,...,...,...,...,...,...,...
995,26-01-2021,120966.11420,32254.19,123470.21880,32921.88,115652.47240,30837.37,121767.12460,32467.77,84972
996,27-01-2021,121753.02310,32464.01,122102.86040,32557.29,109668.14670,29241.72,113885.20900,30366.15,95911
997,28-01-2021,113870.35740,30362.19,126703.43860,33783.98,111919.81180,29842.10,125131.57090,33364.86,92621
998,29-01-2021,125144.02230,33368.18,144510.03780,38531.90,119695.51620,31915.40,128459.45090,34252.20,231827


In [7]:
# 2. Preprocessing
# Convert date to datetime objects
df['date'] = pd.to_datetime(df['date'], dayfirst=True)
df.set_index('date', inplace=True)

In [8]:
# Select relevant variables
data = df[['open_SAR', 'volume']]

# Down-sample to Monthly Data
# We use 'mean' to get the average price and average daily volume per month
monthly_data = pd.DataFrame()
monthly_data['open_SAR'] = data['open_SAR'].resample('M').mean()
monthly_data['volume'] = data['volume'].resample('M').mean()

# Drop any missing values created by resampling
monthly_data.dropna(inplace=True)

print("\n--- Monthly Data Head ---")
print(monthly_data.head())


--- Monthly Data Head ---
                open_SAR        volume
date                                  
2018-05-31  30917.391504  29087.640000
2018-06-30  25513.781179  31407.833333
2018-07-31  26556.535218  35564.354839
2018-08-31  25122.756088  45423.967742
2018-09-30  24775.161152  36688.000000


/tmp/ipython-input-3710290141.py:7: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  monthly_data['open_SAR'] = data['open_SAR'].resample('M').mean()
/tmp/ipython-input-3710290141.py:8: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  monthly_data['volume'] = data['volume'].resample('M').mean()


In [9]:
# 3. Stationarity Check (ADF Test)
def check_stationarity(series, name):
    result = adfuller(series)
    print(f'\nADF Test for {name}:')
    print(f'ADF Statistic: {result[0]:.4f}')
    print(f'p-value: {result[1]:.4f}')
    if result[1] > 0.05:
        print(f"Result: {name} is Non-Stationary (p > 0.05)")
        return False
    else:
        print(f"Result: {name} is Stationary (p < 0.05)")
        return True

is_stationary_sar = check_stationarity(monthly_data['open_SAR'], 'open_SAR')
is_stationary_vol = check_stationarity(monthly_data['volume'], 'volume')


ADF Test for open_SAR:
ADF Statistic: 2.4201
p-value: 0.9990
Result: open_SAR is Non-Stationary (p > 0.05)

ADF Test for volume:
ADF Statistic: -0.1162
p-value: 0.9478
Result: volume is Non-Stationary (p > 0.05)


In [10]:
# 4. Fit ARIMA Models
print("\n" + "="*30)
print("     ARIMA MODEL RESULTS     ")
print("="*30)

def fit_best_arima(series, name, is_stationary):
    best_aic = np.inf
    best_order = None
    best_model = None

    # Determine differencing (d): 1 if non-stationary, else 0
    d = 1 if not is_stationary else 0

    # Simple Grid Search for p and q
    print(f"\nSearching best ARIMA order for {name}...")
    for p in range(3):
        for q in range(3):
            try:
                model = ARIMA(series, order=(p, d, q))
                results = model.fit()
                if results.aic < best_aic:
                    best_aic = results.aic
                    best_order = (p, d, q)
                    best_model = results
            except:
                continue

    print(f"Best Model for {name}: ARIMA{best_order} with AIC: {best_aic:.2f}")
    print(best_model.summary())
    return best_model

arima_sar = fit_best_arima(monthly_data['open_SAR'], 'open_SAR', is_stationary_sar)
arima_vol = fit_best_arima(monthly_data['volume'], 'volume', is_stationary_vol)


     ARIMA MODEL RESULTS     

Searching best ARIMA order for open_SAR...


/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/usr/loc

Best Model for open_SAR: ARIMA(2, 1, 0) with AIC: 682.66
                               SARIMAX Results                                
Dep. Variable:               open_SAR   No. Observations:                   33
Model:                 ARIMA(2, 1, 0)   Log Likelihood                -338.330
Date:                Sat, 17 Jan 2026   AIC                            682.661
Time:                        13:01:36   BIC                            687.058
Sample:                    05-31-2018   HQIC                           684.119
                         - 01-31-2021                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.L1          0.4294      0.187      2.298      0.022       0.063       0.796
ar.L2          0.2097      0.275      0.763      0.446      -0.329       0

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'


In [11]:
# 5. Fit VAR Model
print("\n" + "="*30)
print("      VAR MODEL RESULTS      ")
print("="*30)

# Prepare data for VAR
# If data is non-stationary, we must difference it first
var_data = monthly_data.copy()

if not is_stationary_sar:
    var_data['open_SAR'] = var_data['open_SAR'].diff()
if not is_stationary_vol:
    var_data['volume'] = var_data['volume'].diff()


      VAR MODEL RESULTS      


In [12]:
# Drop NaN values created by differencing
var_data.dropna(inplace=True)

# Initialize VAR model
model_var = VAR(var_data)

# Select Lag Order automatically
lag_selection = model_var.select_order(maxlags=5)
print("\nLag Order Selection:")
print(lag_selection.summary())


Lag Order Selection:
 VAR Order Selection (* highlights the minimums) 
      AIC         BIC         FPE         HQIC   
-------------------------------------------------
0       38.61       38.71   5.889e+16       38.64
1      38.22*      38.51*  3.975e+16*      38.31*
2       38.34       38.82   4.527e+16       38.49
3       38.48       39.15   5.258e+16       38.68
4       38.37       39.23   4.849e+16       38.62
5       38.40       39.45   5.249e+16       38.71
-------------------------------------------------


In [13]:
# Fit the model with the best lag (e.g., lag 1 based on AIC from analysis)
# We let the model pick the best lag based on AIC automatically
results_var = model_var.fit(maxlags=5, ic='aic')

In [14]:
print("\nVAR Model Summary:")
print(results_var.summary())


VAR Model Summary:
  Summary of Regression Results   
Model:                         VAR
Method:                        OLS
Date:           Sat, 17, Jan, 2026
Time:                     13:03:18
--------------------------------------------------------------------
No. of Equations:         2.00000    BIC:                    38.2471
Nobs:                     31.0000    HQIC:                   38.0600
Log likelihood:          -670.502    FPE:                3.09371e+16
AIC:                      37.9695    Det(Omega_mle):     2.57184e+16
--------------------------------------------------------------------
Results for equation open_SAR
                 coefficient       std. error           t-stat            prob
------------------------------------------------------------------------------
const            1661.294151      1550.452522            1.071           0.284
L1.open_SAR         1.042733         0.236975            4.400           0.000
L1.volume           0.003387         0.077301

In [15]:
# Interpretation
print("\n--- Quick Interpretation ---")
print("Check the 'P>|z|' column in the VAR summary above.")
print("- Values < 0.05 indicate a significant relationship.")
print("- 'L1.open_SAR' in the 'open_SAR' equation checks if past price affects current price.")
print("- 'L1.volume' in the 'open_SAR' equation checks if past volume affects current price.")


--- Quick Interpretation ---
Check the 'P>|z|' column in the VAR summary above.
- Values < 0.05 indicate a significant relationship.
- 'L1.open_SAR' in the 'open_SAR' equation checks if past price affects current price.
- 'L1.volume' in the 'open_SAR' equation checks if past volume affects current price.


In [19]:
#Results Summary
#Stationarity: Both Price and Volume were non-stationary. We modeled the changes (first difference) in the data.
#ARIMA (Price): The price data is best modeled by ARIMA(2,1,0). This indicates that the current month's price change is positively correlated with the price changes of the last two months (momentum).
#ARIMA (Volume): The volume data is best modeled by ARIMA(1,1,0).
#VAR Model: The VAR analysis (Order 1) revealed:
#Price Momentum: Past price changes significantly predict future price changes (Coefficient ~ 1.04, t-stat 4.40).
#Independence: Past volume does not significantly predict future price changes (p=0.965), nor do past price changes predict future volume (p=0.179). The two variables appear to move independently on a monthly scale.

In [21]:
#Conclusion
#The analysis indicates that while there is strong internal momentum within the Bitcoin price (SAR) trends (past increases predict future increases), the trading volume does not serve as a significant predictor for price movements in this monthly aggregated dataset. The two time series behave largely independently of each other.